## Primary Issuance Databricks

### Operational/raw schema
`productivity_dev.primaryissuance`

Contains raw ingestion, transformation views, watermark, and search projection:
- `loans_raw`, `bonds_raw`
- `deal_source_current`, `deals_source`
- `deal_tranches_parsed`, `tranches_source`
- `participant/company source and candidate views`
- `pipeline_watermarks`
- `deal_search_projection`

### Consumer parsed schema
`productivity_dev.primaryissuance_parsed`

Contains only six relevant relational tables:
- `deals`
- `tranches`
- `companies`
- `deal_participants`
- `tranche_participants`
- `tranche_participant_roles`

## Initial run order
1. `00_setup.sql`
2. `01_operational_source_views.sql`
3. `02_operational_tranche_views.sql`
4. `03_operational_participant_views.sql`
5. `04_initial_parsed_tables.sql`
6. `06_operational_search_projection.sql`
7. `07_validation.sql`

## Incremental run
After raw ingestion:
1. `05_incremental_merge.sql`
2. `06_operational_search_projection.sql`
3. `07_validation.sql`

Run `05_incremental_merge` in one Databricks SQL session because it uses variables and temporary views. Watermark advances only at end.

### 00_setup

In [0]:
%sql
-- Raw/operational objects stay in primaryissuance.
-- Only consumer relational tables go to primaryissuance_parsed.
CREATE SCHEMA IF NOT EXISTS productivity_dev.primaryissuance;
CREATE SCHEMA IF NOT EXISTS productivity_dev.primaryissuance_parsed;

CREATE TABLE IF NOT EXISTS productivity_dev.primaryissuance.pipeline_watermarks (
  pipeline_name STRING,
  last_processed_at TIMESTAMP,
  updated_at TIMESTAMP
) USING DELTA;

MERGE INTO productivity_dev.primaryissuance.pipeline_watermarks t
USING (SELECT 'primary_issuance_relational' pipeline_name,
              TIMESTAMP '1900-01-01 00:00:00' last_processed_at,
              current_timestamp() updated_at) s
ON t.pipeline_name=s.pipeline_name
WHEN NOT MATCHED THEN INSERT *;


num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


### 01_operational_source_views

In [0]:
%sql
-- Operational latest-source and deal transformation views.
CREATE OR REPLACE VIEW productivity_dev.primaryissuance.deal_source_current AS
WITH u AS (
 SELECT 'Loan' source_instrument_type, document_id,
        try_cast(last_modified_date AS TIMESTAMP) raw_last_modified_date,
        try_cast(document_version AS BIGINT) raw_document_version,
        source_hash, source_json, es_index,
        try_cast(ingested_at AS TIMESTAMP) ingested_at,
        ingestion_run_id, pipeline_version
 FROM productivity_dev.primaryissuance.loans_raw
 UNION ALL
 SELECT 'Bond', document_id,
        try_cast(last_modified_date AS TIMESTAMP),
        try_cast(document_version AS BIGINT),
        source_hash, source_json, es_index,
        try_cast(ingested_at AS TIMESTAMP),
        ingestion_run_id, pipeline_version
 FROM productivity_dev.primaryissuance.bonds_raw
), r AS (
 SELECT *, row_number() OVER (
   PARTITION BY source_instrument_type, document_id
   ORDER BY raw_last_modified_date DESC NULLS LAST,
            raw_document_version DESC NULLS LAST,
            ingested_at DESC NULLS LAST, source_hash DESC) rn
 FROM u
)
SELECT * EXCEPT(rn) FROM r WHERE rn=1;

CREATE OR REPLACE VIEW productivity_dev.primaryissuance.deals_source AS
SELECT
 concat(lower(source_instrument_type),':',cast(try_cast(get_json_object(source_json,'$.id') AS BIGINT) AS STRING)) deal_key,
 try_cast(get_json_object(source_json,'$.id') AS BIGINT) deal_id,
 coalesce(get_json_object(source_json,'$.instrumentType'),source_instrument_type) instrument_type,
 get_json_object(source_json,'$.status') deal_status,
 get_json_object(source_json,'$.debtStatus') debt_status,
 get_json_object(source_json,'$.publishStatus') publish_status,
 get_json_object(source_json,'$.useOfProceeds') primary_use_of_proceeds,
 try_cast(get_json_object(source_json,'$.closeDate') AS DATE) close_date,
 try_cast(get_json_object(source_json,'$.amount.euro') AS DECIMAL(38,6)) amount_euro,
 try_cast(get_json_object(source_json,'$.amount.usd') AS DECIMAL(38,6)) amount_usd,
 try_cast(get_json_object(source_json,'$.debutIssuer') AS BOOLEAN) debut_issuer,
 try_cast(get_json_object(source_json,'$.middleMarket') AS BOOLEAN) middle_market,
 try_cast(get_json_object(source_json,'$.borrower.id') AS BIGINT) primary_borrower_id,
 get_json_object(source_json,'$.borrower.masterHubId') primary_borrower_master_hub_id,
 get_json_object(source_json,'$.borrower.name') primary_borrower_name,
 get_json_object(source_json,'$.borrower.country') primary_borrower_country,
 get_json_object(source_json,'$.borrower.region') primary_borrower_region,
 get_json_object(source_json,'$.borrower.sector') primary_borrower_sector,
 from_json(get_json_object(source_json,'$.borrower.types'),'ARRAY<STRING>') primary_borrower_types,
 try_cast(get_json_object(source_json,'$.borrower.parentCompanyId') AS BIGINT) primary_borrower_parent_company_id,
 get_json_object(source_json,'$.borrower.parentCompanyMasterHubId') primary_borrower_parent_master_hub_id,
 get_json_object(source_json,'$.borrower.parentCompany') primary_borrower_parent_company_name,
 get_json_object(source_json,'$.lastEvent') last_event,
 get_json_object(source_json,'$.lastModifiedBy') last_modified_by,
 try_cast(get_json_object(source_json,'$.lastModifiedDate') AS TIMESTAMP) last_modified_date,
 try_cast(get_json_object(source_json,'$.version') AS BIGINT) document_version,
 get_json_object(source_json,'$.changeComment') change_comment,
 get_json_object(source_json,'$.comments') comments,
 get_json_object(source_json,'$.internalCommentary') internal_commentary,
 from_json(get_json_object(source_json,'$.sources'),'ARRAY<STRUCT<type:STRING>>') sources,
 from_json(get_json_object(source_json,'$.siteEditionTagging'),'ARRAY<STRUCT<edition:STRING,id:STRING,product:STRING>>') site_edition_tagging,
 source_hash, es_index, ingested_at, ingestion_run_id, pipeline_version
FROM productivity_dev.primaryissuance.deal_source_current
WHERE try_cast(get_json_object(source_json,'$.id') AS BIGINT) IS NOT NULL;


### 02_operational_tranche_views

In [0]:
%sql
-- Operational typed nested view and relational tranche source view.
CREATE OR REPLACE VIEW productivity_dev.primaryissuance.deal_tranches_parsed AS
SELECT source_instrument_type,
 try_cast(get_json_object(source_json,'$.id') AS BIGINT) deal_id,
 concat(lower(source_instrument_type),':',cast(try_cast(get_json_object(source_json,'$.id') AS BIGINT) AS STRING)) deal_key,
 source_hash, ingested_at, ingestion_run_id,
 from_json(get_json_object(source_json,'$.tranches'),
 'ARRAY<STRUCT<id:BIGINT,active:BOOLEAN,addOnTap:BOOLEAN,
 amount:STRUCT<original:DECIMAL(38,6),euro:DECIMAL(38,6),usd:DECIMAL(38,6)>,
 currency:STRING,settlementCurrency:STRING,type:STRING,seniority:STRING,secured:BOOLEAN,
 closeDate:STRING,commitmentDate:STRING,maturityDate:STRING,loanSigningDate:STRING,tenor:INT,
 countryOfRisk:STRING,countryOfSyndication:STRING,distributionMethod:STRING,governingLaw:STRING,
 includeInLeagueTable:BOOLEAN,institutional:BOOLEAN,underwritten:BOOLEAN,leveraged:BOOLEAN,
 directLending:BOOLEAN,covenantLite:BOOLEAN,assetBasedLending:BOOLEAN,islamicFinance:BOOLEAN,
 greenFinancing:BOOLEAN,esgFinancing:BOOLEAN,baseRate:STRING,margin:DECIMAL(18,6),
 interest:STRUCT<interestType:STRING,initialPaymentDate:STRING,paymentFrequency:STRING>,
 repaymentType:STRING,coupon:DECIMAL(18,10),yieldToMaturity:DECIMAL(18,10),priceOverOID:DECIMAL(18,10),
 highYield:BOOLEAN,perpetual:BOOLEAN,onshoreGuarantee:BOOLEAN,placementType:STRING,regulation:STRING,
 bondPricingDate:STRING,bondRoadshowDate:STRING,bondSettlementDate:STRING,
 fitchRating:STRING,moodysRating:STRING,spRating:STRING,comments:STRING,
 useOfProceeds:ARRAY<STRING>,bondListing:ARRAY<STRING>,
 additionalBorrowers:ARRAY<STRUCT<id:BIGINT,masterHubId:STRING,name:STRING,country:STRING,region:STRING,sector:STRING,types:ARRAY<STRING>>>,
 associates:ARRAY<STRUCT<id:BIGINT,masterHubId:STRING,name:STRING,role:STRING>>,
 banks:ARRAY<STRUCT<id:BIGINT,masterHubId:STRING,name:STRING,allocationAmount:DECIMAL(38,6),currency:STRING,parentCompany:STRING,parentCompanyId:BIGINT,parentCompanyMasterHubId:STRING,rootParentCompany:STRING,rootParentCompanyId:BIGINT,rootParentCompanyMasterHubId:STRING,roles:ARRAY<STRING>>>,
 lawFirms:ARRAY<STRUCT<company:STRUCT<id:BIGINT,masterHubId:STRING,mmgid:STRING,name:STRING>,laws:ARRAY<STRING>,roles:ARRAY<STRING>>>,
 otherFees:ARRAY<STRUCT<fee:DECIMAL(18,6),type:STRING>>,
 upfrontFees:ARRAY<STRUCT<currency:STRING,maxAmount:DECIMAL(38,6),minAmount:DECIMAL(38,6),title:STRING,upfrontFee:DECIMAL(18,6)>>,
 isin:ARRAY<STRUCT<id:STRING,type:STRING>>,cusip:ARRAY<STRUCT<id:STRING,type:STRING>>>>') tranches
FROM productivity_dev.primaryissuance.deal_source_current;

CREATE OR REPLACE VIEW productivity_dev.primaryissuance.tranches_source AS
SELECT concat(lower(source_instrument_type),':',cast(deal_id AS STRING),':',cast(t.id AS STRING)) tranche_key,
 deal_key,deal_id,source_instrument_type instrument_type,t.id tranche_id,pos tranche_position,
 t.active,t.addOnTap add_on_tap,t.type tranche_type,t.seniority,t.secured,
 t.amount.original amount_original,t.amount.euro amount_euro,t.amount.usd amount_usd,
 t.currency original_currency,t.settlementCurrency settlement_currency,
 try_cast(t.closeDate AS DATE) close_date,try_cast(t.commitmentDate AS DATE) commitment_date,
 try_cast(t.maturityDate AS DATE) maturity_date,try_cast(t.loanSigningDate AS DATE) loan_signing_date,
 t.tenor tenor_months,t.countryOfRisk country_of_risk,t.countryOfSyndication country_of_syndication,
 t.distributionMethod distribution_type,t.governingLaw governing_law,
 t.includeInLeagueTable include_in_league_table,t.institutional,t.underwritten,
 t.assetBasedLending asset_based_lending,t.islamicFinance islamic_finance,t.leveraged,
 t.directLending direct_lending,t.covenantLite covenant_lite,t.baseRate base_rate,t.margin margin_bps,
 t.repaymentType repayment_type,t.highYield high_yield,t.perpetual,t.onshoreGuarantee onshore_guarantee,
 t.placementType placement_type,t.regulation,t.coupon coupon_decimal,t.coupon*100 coupon_percent,
 t.yieldToMaturity yield_to_maturity_decimal,t.yieldToMaturity*100 yield_to_maturity_percent,
 t.priceOverOID price_over_oid,try_cast(t.bondPricingDate AS DATE) bond_pricing_date,
 try_cast(t.bondRoadshowDate AS DATE) bond_roadshow_date,try_cast(t.bondSettlementDate AS DATE) bond_settlement_date,
 t.fitchRating fitch_rating,t.moodysRating moodys_rating,t.spRating sp_rating,
 t.interest.interestType interest_type,try_cast(t.interest.initialPaymentDate AS DATE) initial_interest_payment_date,
 t.interest.paymentFrequency interest_payment_frequency,t.useOfProceeds use_of_proceeds,t.bondListing bond_listings,
 t.greenFinancing green_financing,t.esgFinancing esg_financing,
 CASE WHEN t.greenFinancing OR t.esgFinancing THEN true
      WHEN t.greenFinancing=false AND t.esgFinancing=false THEN false END green_esg_financing,
 t.comments,source_hash,ingested_at,ingestion_run_id
FROM productivity_dev.primaryissuance.deal_tranches_parsed
LATERAL VIEW OUTER posexplode(tranches) e AS pos,t
WHERE t IS NOT NULL AND t.id IS NOT NULL;


### 03_operational_participant_views

In [0]:
%sql
-- Operational participant/company transformation views.
CREATE OR REPLACE VIEW productivity_dev.primaryissuance.deal_participants_source AS
SELECT sha2(concat_ws('|',deal_key,CASE WHEN instrument_type='Bond' THEN 'Issuer' ELSE 'Borrower' END,
 coalesce(primary_borrower_master_hub_id,cast(primary_borrower_id AS STRING),primary_borrower_name)),256) deal_participant_key,
 deal_key,deal_id,instrument_type,
 CASE WHEN primary_borrower_master_hub_id IS NOT NULL THEN concat('masterhub:',primary_borrower_master_hub_id)
      WHEN primary_borrower_id IS NOT NULL THEN concat('source:',cast(primary_borrower_id AS STRING)) END company_key,
 primary_borrower_id source_company_id,primary_borrower_master_hub_id master_hub_id,primary_borrower_name company_name,
 CASE WHEN instrument_type='Bond' THEN 'Issuer' ELSE 'Borrower' END involvement,
 'Primary Borrower' source_involvement,true is_primary,
 primary_borrower_country participant_country,primary_borrower_region participant_region,
 primary_borrower_sector participant_industry,primary_borrower_types participant_company_types,
 primary_borrower_parent_company_id parent_source_company_id,
 primary_borrower_parent_master_hub_id parent_master_hub_id,
 primary_borrower_parent_company_name parent_company_name,
 source_hash,last_modified_date,ingested_at
FROM productivity_dev.primaryissuance.deals_source
WHERE primary_borrower_id IS NOT NULL OR primary_borrower_master_hub_id IS NOT NULL OR primary_borrower_name IS NOT NULL;

CREATE OR REPLACE VIEW productivity_dev.primaryissuance.tranche_participant_candidates AS
SELECT deal_key,deal_id,source_instrument_type instrument_type,
 concat(lower(source_instrument_type),':',cast(deal_id AS STRING),':',cast(t.id AS STRING)) tranche_key,
 t.id tranche_id,pos participant_position,'Additional Borrower' participant_type,
 CASE WHEN x.masterHubId IS NOT NULL THEN concat('masterhub:',x.masterHubId) ELSE concat('source:',cast(x.id AS STRING)) END company_key,
 x.id source_company_id,x.masterHubId master_hub_id,cast(NULL AS STRING) mmgid,x.name company_name,
 x.country participant_country,x.region participant_region,x.sector participant_industry,x.types participant_company_types,
 cast(NULL AS DECIMAL(38,6)) allocation_amount,cast(NULL AS STRING) allocation_currency,
 cast(NULL AS BIGINT) parent_source_company_id,cast(NULL AS STRING) parent_master_hub_id,cast(NULL AS STRING) parent_company_name,
 cast(NULL AS BIGINT) root_parent_source_company_id,cast(NULL AS STRING) root_parent_master_hub_id,cast(NULL AS STRING) root_parent_company_name,
 cast(NULL AS ARRAY<STRING>) applicable_laws,array('Additional Borrower') roles,source_hash,ingested_at,ingestion_run_id
FROM productivity_dev.primaryissuance.deal_tranches_parsed
LATERAL VIEW explode(tranches) a AS t LATERAL VIEW posexplode(t.additionalBorrowers) b AS pos,x
UNION ALL
SELECT deal_key,deal_id,source_instrument_type,
 concat(lower(source_instrument_type),':',cast(deal_id AS STRING),':',cast(t.id AS STRING)),t.id,pos,'Bank',
 CASE WHEN x.masterHubId IS NOT NULL THEN concat('masterhub:',x.masterHubId) ELSE concat('source:',cast(x.id AS STRING)) END,
 x.id,x.masterHubId,cast(NULL AS STRING),x.name,cast(NULL AS STRING),cast(NULL AS STRING),cast(NULL AS STRING),cast(NULL AS ARRAY<STRING>),
 x.allocationAmount,x.currency,x.parentCompanyId,x.parentCompanyMasterHubId,x.parentCompany,
 x.rootParentCompanyId,x.rootParentCompanyMasterHubId,x.rootParentCompany,cast(NULL AS ARRAY<STRING>),x.roles,source_hash,ingested_at,ingestion_run_id
FROM productivity_dev.primaryissuance.deal_tranches_parsed
LATERAL VIEW explode(tranches) a AS t LATERAL VIEW posexplode(t.banks) b AS pos,x
UNION ALL
SELECT deal_key,deal_id,source_instrument_type,
 concat(lower(source_instrument_type),':',cast(deal_id AS STRING),':',cast(t.id AS STRING)),t.id,pos,'Associate',
 CASE WHEN x.masterHubId IS NOT NULL THEN concat('masterhub:',x.masterHubId) ELSE concat('source:',cast(x.id AS STRING)) END,
 x.id,x.masterHubId,cast(NULL AS STRING),x.name,cast(NULL AS STRING),cast(NULL AS STRING),cast(NULL AS STRING),cast(NULL AS ARRAY<STRING>),
 cast(NULL AS DECIMAL(38,6)),cast(NULL AS STRING),cast(NULL AS BIGINT),cast(NULL AS STRING),cast(NULL AS STRING),
 cast(NULL AS BIGINT),cast(NULL AS STRING),cast(NULL AS STRING),cast(NULL AS ARRAY<STRING>),array(x.role),source_hash,ingested_at,ingestion_run_id
FROM productivity_dev.primaryissuance.deal_tranches_parsed
LATERAL VIEW explode(tranches) a AS t LATERAL VIEW posexplode(t.associates) b AS pos,x
UNION ALL
SELECT deal_key,deal_id,source_instrument_type,
 concat(lower(source_instrument_type),':',cast(deal_id AS STRING),':',cast(t.id AS STRING)),t.id,pos,'Law Firm',
 CASE WHEN x.company.masterHubId IS NOT NULL THEN concat('masterhub:',x.company.masterHubId)
      WHEN x.company.mmgid IS NOT NULL THEN concat('mmgid:',x.company.mmgid)
      ELSE concat('source:',cast(x.company.id AS STRING)) END,
 x.company.id,x.company.masterHubId,x.company.mmgid,x.company.name,cast(NULL AS STRING),cast(NULL AS STRING),cast(NULL AS STRING),cast(NULL AS ARRAY<STRING>),
 cast(NULL AS DECIMAL(38,6)),cast(NULL AS STRING),cast(NULL AS BIGINT),cast(NULL AS STRING),cast(NULL AS STRING),
 cast(NULL AS BIGINT),cast(NULL AS STRING),cast(NULL AS STRING),x.laws,x.roles,source_hash,ingested_at,ingestion_run_id
FROM productivity_dev.primaryissuance.deal_tranches_parsed
LATERAL VIEW explode(tranches) a AS t LATERAL VIEW posexplode(t.lawFirms) b AS pos,x;

CREATE OR REPLACE VIEW productivity_dev.primaryissuance.tranche_participants_source AS
WITH p AS (SELECT sha2(concat_ws('|',tranche_key,participant_type,coalesce(company_key,company_name),
 CASE WHEN participant_type='Associate' THEN concat_ws(',',array_sort(filter(roles,r -> r IS NOT NULL))) ELSE '' END),256) tranche_participant_key,*
 FROM productivity_dev.primaryissuance.tranche_participant_candidates
 WHERE company_key IS NOT NULL OR company_name IS NOT NULL),
r AS (SELECT *,row_number() OVER(PARTITION BY tranche_participant_key ORDER BY CASE WHEN allocation_amount IS NOT NULL THEN 0 ELSE 1 END,participant_position,ingested_at DESC) rn FROM p)
SELECT tranche_participant_key,deal_key,deal_id,tranche_key,tranche_id,instrument_type,company_key,source_company_id,master_hub_id,mmgid,company_name,
 participant_type,participant_position,participant_country,participant_region,participant_industry,participant_company_types,allocation_amount,allocation_currency,
 parent_source_company_id,parent_master_hub_id,parent_company_name,
 CASE WHEN parent_master_hub_id IS NOT NULL THEN concat('masterhub:',parent_master_hub_id) WHEN parent_source_company_id IS NOT NULL THEN concat('source:',cast(parent_source_company_id AS STRING)) END parent_company_key,
 root_parent_source_company_id,root_parent_master_hub_id,root_parent_company_name,
 CASE WHEN root_parent_master_hub_id IS NOT NULL THEN concat('masterhub:',root_parent_master_hub_id) WHEN root_parent_source_company_id IS NOT NULL THEN concat('source:',cast(root_parent_source_company_id AS STRING)) END root_parent_company_key,
 applicable_laws,roles,source_hash,ingested_at,ingestion_run_id FROM r WHERE rn=1;

CREATE OR REPLACE VIEW productivity_dev.primaryissuance.tranche_participant_roles_source AS
SELECT sha2(concat_ws('|',tranche_participant_key,role),256) tranche_participant_role_key,
 tranche_participant_key,deal_key,deal_id,tranche_key,tranche_id,instrument_type,company_key,source_company_id,company_name,participant_type,
 pos role_position,role involvement,
 CASE WHEN participant_type='Bank' THEN 'Lender' WHEN participant_type='Law Firm' THEN 'Law Firm'
      WHEN participant_type='Additional Borrower' THEN 'Borrower' ELSE role END involvement_category,
 source_hash,ingested_at
FROM productivity_dev.primaryissuance.tranche_participants_source
LATERAL VIEW posexplode(roles) x AS pos,role WHERE role IS NOT NULL AND trim(role)<>'';

CREATE OR REPLACE VIEW productivity_dev.primaryissuance.company_candidates AS
SELECT company_key,source_company_id,master_hub_id,cast(NULL AS STRING) mmgid,company_name,
 participant_country country,participant_region region,participant_industry industry,participant_company_types company_types,
 parent_source_company_id,parent_master_hub_id,parent_company_name,cast(NULL AS BIGINT) root_parent_source_company_id,
 cast(NULL AS STRING) root_parent_master_hub_id,cast(NULL AS STRING) root_parent_company_name,
 deal_key,cast(NULL AS STRING) tranche_key,'Primary Borrower' company_source_type,last_modified_date,source_hash,ingested_at
FROM productivity_dev.primaryissuance.deal_participants_source
UNION ALL
SELECT company_key,source_company_id,master_hub_id,mmgid,company_name,participant_country,participant_region,participant_industry,participant_company_types,
 parent_source_company_id,parent_master_hub_id,parent_company_name,root_parent_source_company_id,root_parent_master_hub_id,root_parent_company_name,
 deal_key,tranche_key,participant_type,cast(NULL AS TIMESTAMP),source_hash,ingested_at
FROM productivity_dev.primaryissuance.tranche_participants_source;

CREATE OR REPLACE VIEW productivity_dev.primaryissuance.companies_source AS
WITH r AS (SELECT *,row_number() OVER(PARTITION BY company_key ORDER BY
 CASE company_source_type WHEN 'Primary Borrower' THEN 1 WHEN 'Additional Borrower' THEN 2 WHEN 'Bank' THEN 3 WHEN 'Associate' THEN 4 WHEN 'Law Firm' THEN 5 ELSE 99 END,
 CASE WHEN country IS NOT NULL THEN 0 ELSE 1 END,CASE WHEN industry IS NOT NULL THEN 0 ELSE 1 END,ingested_at DESC) rn
 FROM productivity_dev.primaryissuance.company_candidates WHERE company_key IS NOT NULL)
SELECT company_key,source_company_id,master_hub_id,mmgid,company_name,country,region,industry,company_types,
 parent_source_company_id,parent_master_hub_id,parent_company_name,
 CASE WHEN parent_master_hub_id IS NOT NULL THEN concat('masterhub:',parent_master_hub_id) WHEN parent_source_company_id IS NOT NULL THEN concat('source:',cast(parent_source_company_id AS STRING)) END parent_company_key,
 root_parent_source_company_id,root_parent_master_hub_id,root_parent_company_name,
 CASE WHEN root_parent_master_hub_id IS NOT NULL THEN concat('masterhub:',root_parent_master_hub_id) WHEN root_parent_source_company_id IS NOT NULL THEN concat('source:',cast(root_parent_source_company_id AS STRING)) END root_parent_company_key,
 company_source_type selected_source_type,source_hash,ingested_at FROM r WHERE rn=1;


### 04_initial_parsed_tables

In [0]:
%sql
-- Only these six consumer-facing tables are created in parsed schema.
CREATE OR REPLACE TABLE productivity_dev.primaryissuance_parsed.deals USING DELTA AS
SELECT * FROM productivity_dev.primaryissuance.deals_source;
CREATE OR REPLACE TABLE productivity_dev.primaryissuance_parsed.tranches USING DELTA AS
SELECT * FROM productivity_dev.primaryissuance.tranches_source;
CREATE OR REPLACE TABLE productivity_dev.primaryissuance_parsed.deal_participants USING DELTA AS
SELECT * FROM productivity_dev.primaryissuance.deal_participants_source;
CREATE OR REPLACE TABLE productivity_dev.primaryissuance_parsed.tranche_participants USING DELTA AS
SELECT * FROM productivity_dev.primaryissuance.tranche_participants_source;
CREATE OR REPLACE TABLE productivity_dev.primaryissuance_parsed.tranche_participant_roles USING DELTA AS
SELECT * FROM productivity_dev.primaryissuance.tranche_participant_roles_source;
CREATE OR REPLACE TABLE productivity_dev.primaryissuance_parsed.companies USING DELTA AS
SELECT * FROM productivity_dev.primaryissuance.companies_source;
UPDATE productivity_dev.primaryissuance.pipeline_watermarks
SET last_processed_at=current_timestamp(),updated_at=current_timestamp()
WHERE pipeline_name='primary_issuance_relational';


num_affected_rows
1


### 05_incremental_merge

In [0]:
%sql
-- Run entire file in one Databricks SQL session.
DECLARE OR REPLACE VARIABLE from_ts TIMESTAMP;
DECLARE OR REPLACE VARIABLE to_ts TIMESTAMP;
SET VAR from_ts=(SELECT last_processed_at FROM productivity_dev.primaryissuance.pipeline_watermarks WHERE pipeline_name='primary_issuance_relational');
SET VAR to_ts=current_timestamp();

CREATE OR REPLACE TEMP VIEW affected_deals AS
SELECT DISTINCT concat('loan:',cast(try_cast(get_json_object(source_json,'$.id') AS BIGINT) AS STRING)) deal_key
FROM productivity_dev.primaryissuance.loans_raw WHERE try_cast(ingested_at AS TIMESTAMP)>from_ts AND try_cast(ingested_at AS TIMESTAMP)<=to_ts
UNION
SELECT DISTINCT concat('bond:',cast(try_cast(get_json_object(source_json,'$.id') AS BIGINT) AS STRING))
FROM productivity_dev.primaryissuance.bonds_raw WHERE try_cast(ingested_at AS TIMESTAMP)>from_ts AND try_cast(ingested_at AS TIMESTAMP)<=to_ts;

CREATE OR REPLACE TEMP VIEW d_i AS SELECT s.* FROM productivity_dev.primaryissuance.deals_source s JOIN affected_deals a USING(deal_key);
CREATE OR REPLACE TEMP VIEW t_i AS SELECT s.* FROM productivity_dev.primaryissuance.tranches_source s JOIN affected_deals a USING(deal_key);
CREATE OR REPLACE TEMP VIEW dp_i AS SELECT s.* FROM productivity_dev.primaryissuance.deal_participants_source s JOIN affected_deals a USING(deal_key);
CREATE OR REPLACE TEMP VIEW tp_i AS SELECT s.* FROM productivity_dev.primaryissuance.tranche_participants_source s JOIN affected_deals a USING(deal_key);
CREATE OR REPLACE TEMP VIEW r_i AS SELECT s.* FROM productivity_dev.primaryissuance.tranche_participant_roles_source s JOIN affected_deals a USING(deal_key);
CREATE OR REPLACE TEMP VIEW c_keys AS SELECT company_key FROM dp_i WHERE company_key IS NOT NULL UNION SELECT company_key FROM tp_i WHERE company_key IS NOT NULL;
CREATE OR REPLACE TEMP VIEW c_i AS SELECT s.* FROM productivity_dev.primaryissuance.companies_source s JOIN c_keys k USING(company_key);

MERGE INTO productivity_dev.primaryissuance_parsed.deals t USING d_i s ON t.deal_key=s.deal_key WHEN MATCHED AND NOT(t.source_hash<=>s.source_hash) THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *;
DELETE FROM productivity_dev.primaryissuance_parsed.deals t WHERE EXISTS(SELECT 1 FROM affected_deals a WHERE a.deal_key=t.deal_key) AND NOT EXISTS(SELECT 1 FROM d_i s WHERE s.deal_key=t.deal_key);
MERGE INTO productivity_dev.primaryissuance_parsed.tranches t USING t_i s ON t.tranche_key=s.tranche_key WHEN MATCHED AND NOT(t.source_hash<=>s.source_hash) THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *;
DELETE FROM productivity_dev.primaryissuance_parsed.tranches t WHERE EXISTS(SELECT 1 FROM affected_deals a WHERE a.deal_key=t.deal_key) AND NOT EXISTS(SELECT 1 FROM t_i s WHERE s.tranche_key=t.tranche_key);
MERGE INTO productivity_dev.primaryissuance_parsed.deal_participants t USING dp_i s ON t.deal_participant_key=s.deal_participant_key WHEN MATCHED AND NOT(t.source_hash<=>s.source_hash) THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *;
DELETE FROM productivity_dev.primaryissuance_parsed.deal_participants t WHERE EXISTS(SELECT 1 FROM affected_deals a WHERE a.deal_key=t.deal_key) AND NOT EXISTS(SELECT 1 FROM dp_i s WHERE s.deal_participant_key=t.deal_participant_key);
MERGE INTO productivity_dev.primaryissuance_parsed.tranche_participants t USING tp_i s ON t.tranche_participant_key=s.tranche_participant_key WHEN MATCHED AND NOT(t.source_hash<=>s.source_hash) THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *;
DELETE FROM productivity_dev.primaryissuance_parsed.tranche_participants t WHERE EXISTS(SELECT 1 FROM affected_deals a WHERE a.deal_key=t.deal_key) AND NOT EXISTS(SELECT 1 FROM tp_i s WHERE s.tranche_participant_key=t.tranche_participant_key);
MERGE INTO productivity_dev.primaryissuance_parsed.tranche_participant_roles t USING r_i s ON t.tranche_participant_role_key=s.tranche_participant_role_key WHEN MATCHED AND NOT(t.source_hash<=>s.source_hash) THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *;
DELETE FROM productivity_dev.primaryissuance_parsed.tranche_participant_roles t WHERE EXISTS(SELECT 1 FROM affected_deals a WHERE a.deal_key=t.deal_key) AND NOT EXISTS(SELECT 1 FROM r_i s WHERE s.tranche_participant_role_key=t.tranche_participant_role_key);
MERGE INTO productivity_dev.primaryissuance_parsed.companies t USING c_i s ON t.company_key=s.company_key WHEN MATCHED AND NOT(t.source_hash<=>s.source_hash) THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *;
DELETE FROM productivity_dev.primaryissuance_parsed.companies c WHERE NOT EXISTS(SELECT 1 FROM productivity_dev.primaryissuance_parsed.deal_participants p WHERE p.company_key=c.company_key) AND NOT EXISTS(SELECT 1 FROM productivity_dev.primaryissuance_parsed.tranche_participants p WHERE p.company_key=c.company_key);

MERGE INTO productivity_dev.primaryissuance.pipeline_watermarks t
USING(SELECT 'primary_issuance_relational' pipeline_name,to_ts last_processed_at,current_timestamp() updated_at)s
ON t.pipeline_name=s.pipeline_name WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *;


num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
1,1,0,0


### 06_operational_search_projection

In [0]:
%sql
-- Search projection intentionally stays in operational schema.
CREATE OR REPLACE TABLE productivity_dev.primaryissuance.deal_search_projection USING DELTA AS
WITH dp AS (SELECT deal_key,array_sort(array_distinct(collect_list(company_name))) deal_company_names,
 array_sort(array_distinct(collect_list(involvement))) deal_involvements,
 array_sort(array_distinct(flatten(collect_list(participant_company_types)))) borrower_types,
 array_sort(array_distinct(collect_list(participant_industry))) borrower_industries,
 array_sort(array_distinct(collect_list(participant_country))) borrower_countries,
 array_sort(array_distinct(collect_list(participant_region))) borrower_regions
 FROM productivity_dev.primaryissuance_parsed.deal_participants GROUP BY deal_key),
ta AS (SELECT deal_key,array_sort(array_distinct(collect_list(tranche_type))) tranche_types,
 array_sort(array_distinct(collect_list(seniority))) seniorities,array_sort(array_distinct(collect_list(country_of_risk))) countries_of_risk,
 array_sort(array_distinct(collect_list(country_of_syndication))) countries_of_syndication,array_sort(array_distinct(collect_list(original_currency))) original_currencies,
 min(commitment_date) earliest_commitment_date,max(commitment_date) latest_commitment_date,min(maturity_date) earliest_maturity_date,max(maturity_date) latest_maturity_date,
 min(tenor_months) minimum_tenor_months,max(tenor_months) maximum_tenor_months,min(margin_bps) minimum_margin_bps,max(margin_bps) maximum_margin_bps,
 min(coupon_percent) minimum_coupon_percent,max(coupon_percent) maximum_coupon_percent,
 max(CASE WHEN high_yield THEN 1 ELSE 0 END)=1 has_high_yield_tranche,max(CASE WHEN leveraged THEN 1 ELSE 0 END)=1 has_leveraged_tranche,
 max(CASE WHEN direct_lending THEN 1 ELSE 0 END)=1 has_direct_lending_tranche,max(CASE WHEN green_esg_financing THEN 1 ELSE 0 END)=1 has_green_esg_tranche
 FROM productivity_dev.primaryissuance_parsed.tranches GROUP BY deal_key),
tp AS (SELECT deal_key,array_sort(array_distinct(collect_list(CASE WHEN participant_type='Bank' THEN company_name END))) lenders,
 array_sort(array_distinct(collect_list(CASE WHEN participant_type='Law Firm' THEN company_name END))) law_firms
 FROM productivity_dev.primaryissuance_parsed.tranche_participants GROUP BY deal_key)
SELECT d.*,dp.* EXCEPT(deal_key),ta.* EXCEPT(deal_key),tp.* EXCEPT(deal_key)
FROM productivity_dev.primaryissuance_parsed.deals d
LEFT JOIN dp USING(deal_key) LEFT JOIN ta USING(deal_key) LEFT JOIN tp USING(deal_key);


num_affected_rows,num_inserted_rows


### 07_validation

In [0]:
%sql
-- Parsed schema must contain only six consumer tables.
SHOW TABLES IN productivity_dev.primaryissuance_parsed;

SELECT deal_key,count(*) c FROM productivity_dev.primaryissuance_parsed.deals GROUP BY deal_key HAVING c>1;
SELECT tranche_key,count(*) c FROM productivity_dev.primaryissuance_parsed.tranches GROUP BY tranche_key HAVING c>1;
SELECT deal_participant_key,count(*) c FROM productivity_dev.primaryissuance_parsed.deal_participants GROUP BY deal_participant_key HAVING c>1;
SELECT tranche_participant_key,count(*) c FROM productivity_dev.primaryissuance_parsed.tranche_participants GROUP BY tranche_participant_key HAVING c>1;
SELECT tranche_participant_role_key,count(*) c FROM productivity_dev.primaryissuance_parsed.tranche_participant_roles GROUP BY tranche_participant_role_key HAVING c>1;

SELECT t.tranche_key FROM productivity_dev.primaryissuance_parsed.tranches t LEFT JOIN productivity_dev.primaryissuance_parsed.deals d USING(deal_key) WHERE d.deal_key IS NULL;
SELECT p.tranche_participant_key FROM productivity_dev.primaryissuance_parsed.tranche_participants p LEFT JOIN productivity_dev.primaryissuance_parsed.tranches t USING(tranche_key) WHERE t.tranche_key IS NULL;
SELECT r.tranche_participant_role_key FROM productivity_dev.primaryissuance_parsed.tranche_participant_roles r LEFT JOIN productivity_dev.primaryissuance_parsed.tranche_participants p USING(tranche_participant_key) WHERE p.tranche_participant_key IS NULL;

SELECT * FROM productivity_dev.primaryissuance.pipeline_watermarks WHERE pipeline_name='primary_issuance_relational';


pipeline_name,last_processed_at,updated_at
primary_issuance_relational,2026-09-30T05:30:11.20979Z,2026-09-30T05:31:15.223832Z
